- copy of Vendants machine learning notebook
- copied on 29.07.2026

In [ ]:
from pathlib import Path
import xarray as xr



import numpy as np



import tensorflow as tf

from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

## Building a model Neural Network 

In [ ]:
print(rad_1d.shape, rad_3d.shape, delta.shape)

In [ ]:
# --------------------------------------------------
# 2. Normalize data
# --------------------------------------------------

x_mean = np.nanmean(rad_1d)
x_std  = np.nanstd(rad_1d)

y_mean = np.nanmean(delta)
y_std  = np.nanstd(delta)

rad_1d_norm = (rad_1d - x_mean) / x_std
delta_norm  = (delta - y_mean) / y_std

rad_1d_norm = np.nan_to_num(rad_1d_norm)
delta_norm  = np.nan_to_num(delta_norm)

In [ ]:
# --------------------------------------------------
# 3. Make image patches
# --------------------------------------------------

def make_patches(x, y, patch_size=64, stride=32):
    X_patches = []
    Y_patches = []

    ny, nx = x.shape

    for i in range(0, ny - patch_size + 1, stride):
        for j in range(0, nx - patch_size + 1, stride):
            x_patch = x[i:i+patch_size, j:j+patch_size]
            y_patch = y[i:i+patch_size, j:j+patch_size]

            X_patches.append(x_patch[..., np.newaxis])
            Y_patches.append(y_patch[..., np.newaxis])

    return np.array(X_patches, dtype=np.float32), np.array(Y_patches, dtype=np.float32)


PATCH_SIZE = 64
STRIDE = 32

X, Y = make_patches(rad_1d_norm, delta_norm, PATCH_SIZE, STRIDE)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

In [ ]:
# --------------------------------------------------
# 4. Train/validation split
# --------------------------------------------------

n = X.shape[0]
idx = np.random.permutation(n)

train_frac = 0.8
n_train = int(train_frac * n)

train_idx = idx[:n_train]
val_idx   = idx[n_train:]

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val     = X[val_idx], Y[val_idx]

print("Train:", X_train.shape, Y_train.shape)
print("Val:", X_val.shape, Y_val.shape)

In [ ]:
# --------------------------------------------------
# 5. Build small CNN
# --------------------------------------------------

def build_model(patch_size):
    inp = layers.Input(shape=(patch_size, patch_size, 1))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)

    out = layers.Conv2D(1, 1, padding="same", activation="linear")(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model


model = build_model(PATCH_SIZE)
model.summary()

In [ ]:
# --------------------------------------------------
# 6. Train model
# --------------------------------------------------

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks
)

In [ ]:
# --------------------------------------------------
# 7. Plot training curve
# --------------------------------------------------

plt.figure(figsize=(7,5))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.title("Training curve")
plt.show()

In [ ]:
# --------------------------------------------------
# 8. Predict one validation patch
# --------------------------------------------------

k = 0

x_patch = X_val[k:k+1]
y_true_norm = Y_val[k, :, :, 0]

y_pred_norm = model.predict(x_patch)[0, :, :, 0]

y_true = y_true_norm * y_std + y_mean
y_pred = y_pred_norm * y_std + y_mean

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(x_patch[0, :, :, 0], origin="lower")
axes[0].set_title("Input 1D radiance norm")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(y_true, origin="lower", cmap="RdBu_r")
axes[1].set_title("True 3D - 1D")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(y_pred, origin="lower", cmap="RdBu_r")
axes[2].set_title("Predicted 3D - 1D")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------
# 9. Predict full image using patches
# --------------------------------------------------

def predict_full_image(model, x_norm, patch_size=64, stride=32):
    ny, nx = x_norm.shape

    pred_sum = np.zeros((ny, nx), dtype=np.float32)
    pred_count = np.zeros((ny, nx), dtype=np.float32)

    for i in range(0, ny - patch_size + 1, stride):
        for j in range(0, nx - patch_size + 1, stride):
            patch = x_norm[i:i+patch_size, j:j+patch_size]
            patch = patch[np.newaxis, ..., np.newaxis].astype(np.float32)

            pred_patch = model.predict(patch, verbose=0)[0, :, :, 0]

            pred_sum[i:i+patch_size, j:j+patch_size] += pred_patch
            pred_count[i:i+patch_size, j:j+patch_size] += 1

    pred_norm = pred_sum / np.maximum(pred_count, 1)

    return pred_norm


pred_delta_norm = predict_full_image(
    model,
    rad_1d_norm,
    patch_size=PATCH_SIZE,
    stride=STRIDE
)

pred_delta = pred_delta_norm * y_std + y_mean

pred_3d = rad_1d + pred_delta

In [ ]:
# --------------------------------------------------
# 10. Plot full-field result
# --------------------------------------------------

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

im0 = axes[0].imshow(rad_1d, origin="lower")
axes[0].set_title("1D radiance")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(rad_3d, origin="lower")
axes[1].set_title("True 3D radiance")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(pred_3d, origin="lower")
axes[2].set_title("Predicted 3D radiance")
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(pred_delta - delta, origin="lower", cmap="RdBu_r")
axes[3].set_title("Error: predicted Δ - true Δ")
plt.colorbar(im3, ax=axes[3])

plt.tight_layout()
plt.show()

### Neural Network For ds2

In [ ]:
ds2

In [ ]:
# Remove phy dimension if present
if "phy" in ds2.dims and ds2.sizes["phy"] == 1:
    ds_ml = ds2.squeeze("phy", drop=True)
else:
    ds_ml = ds2

# 1D and 3D radiation fields
eglo_1d = ds_ml["eglo"].sel(exp="1D")
eglo_3d = ds_ml["eglo"].sel(exp="3D")

edir_1d = ds_ml["edir"].sel(exp="1D")
edn_1d  = ds_ml["edn"].sel(exp="1D")
eup_1d  = ds_ml["eup"].sel(exp="1D")

# Target correction
#delta_eglo = eglo_3d - eglo_1d
delta_eglo = ds_ml["eglo"].sel(exp="3D-1D") # only change i did

In [ ]:
# Build cloud inputs

z = ds_ml["z"]
dz = np.abs(np.gradient(z.values))

dz_da = xr.DataArray(
    dz,
    dims=("z",),
    coords={"z": z}
)

# Liquid water path
lwp = (ds_ml["lwc"] * dz_da).sum("z")


# 2D cloud mask
cloud_mask = (ds_ml["masks"].max("z") > 0).astype(float)

In [ ]:
print(ds_ml["lwc"].attrs.get("units"))

In [ ]:
# Stack input channels

input_vars = [
    eglo_1d,
    edir_1d,
    edn_1d,
    eup_1d,
    lwp
]

X = np.stack([v.values for v in input_vars], axis=-1)
Y = delta_eglo.values[..., np.newaxis]

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

In [ ]:
# Split by time

n_time = X.shape[0]
indices = np.arange(n_time)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val     = X[val_idx], Y[val_idx]
X_test, Y_test   = X[test_idx], Y[test_idx]

print("Train:", X_train.shape, Y_train.shape)
print("Val:", X_val.shape, Y_val.shape)
print("Test:", X_test.shape, Y_test.shape)

In [ ]:
# Normalize using training data only

x_mean = X_train.mean(axis=(0, 1, 2), keepdims=True)
x_std  = X_train.std(axis=(0, 1, 2), keepdims=True) + 1e-8

y_mean = Y_train.mean(axis=(0, 1, 2), keepdims=True)
y_std  = Y_train.std(axis=(0, 1, 2), keepdims=True) + 1e-8

X_train_n = (X_train - x_mean) / x_std
X_val_n   = (X_val   - x_mean) / x_std
X_test_n  = (X_test  - x_mean) / x_std

Y_train_n = (Y_train - y_mean) / y_std
Y_val_n   = (Y_val   - y_mean) / y_std
Y_test_n  = (Y_test  - y_mean) / y_std

In [ ]:
# Build CNN

def build_correction_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),

        layers.Conv2D(32, 3, padding="same", activation="relu"),

        layers.Conv2D(1, 1, padding="same", activation="linear")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model


model = build_correction_cnn(X_train_n.shape[1:])
model.summary()

In [ ]:
# Train

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train_n,
    Y_train_n,
    validation_data=(X_val_n, Y_val_n),
    epochs=150,
    batch_size=8,
    callbacks=callbacks
)

In [ ]:
# Evaluate

test_loss, test_mae = model.evaluate(X_test_n, Y_test_n)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)


In [ ]:
# Plot one test prediction

i = 0

pred_delta_n = model.predict(X_test_n[i:i+1])[0]
true_delta_n = Y_test_n[i]

pred_delta = pred_delta_n * y_std.squeeze() + y_mean.squeeze()
true_delta = true_delta_n * y_std.squeeze() + y_mean.squeeze()

eglo_1d_scene = X_test[i, :, :, 0]

error = pred_delta[:, :, 0] - true_delta[:, :, 0]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

im0 = axes[0].imshow(eglo_1d_scene, origin="lower")
axes[0].set_title("1D $E_{glo}$")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(true_delta[:, :, 0], origin="lower", cmap="RdBu_r")
axes[1].set_title("True correction: 3D - 1D")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(pred_delta[:, :, 0], origin="lower", cmap="RdBu_r")
axes[2].set_title("Predicted correction")
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(error, origin="lower", cmap="RdBu_r")
axes[3].set_title("Prediction error")
plt.colorbar(im3, ax=axes[3])

plt.tight_layout()
plt.show()


# U-Net architecture 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [ ]:
def conv_block(x, filters, dropout=0.0):

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x = layers.LayerNormalization(axis=-1)(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x = layers.LayerNormalization(axis=-1)(x)
    x = layers.Activation("relu")(x)

    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    return x

In [ ]:
def build_unet(input_shape):

    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(
        inputs,
        filters=32,
        dropout=0.0
    )
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(
        p1,
        filters=64,
        dropout=0.0
    )
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    b = conv_block(
        p2,
        filters=128,
        dropout=0.1
    )

    # Decoder
    u2 = layers.Conv2DTranspose(
        filters=64,
        kernel_size=2,
        strides=2,
        padding="same"
    )(b)

    u2 = layers.Concatenate()([u2, c2])

    c3 = conv_block(
        u2,
        filters=64,
        dropout=0.0
    )

    u1 = layers.Conv2DTranspose(
        filters=32,
        kernel_size=2,
        strides=2,
        padding="same"
    )(c3)

    u1 = layers.Concatenate()([u1, c1])

    c4 = conv_block(
        u1,
        filters=32,
        dropout=0.0
    )

    outputs = layers.Conv2D(
        filters=1,
        kernel_size=1,
        activation="linear"
    )(c4)

    model = Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=3e-4
        ),

        loss=tf.keras.losses.Huber(delta=1.0),

        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mean_absolute_error"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="root_mean_squared_error"
            )
        ]
    )

    return model

In [ ]:
unet = build_unet(X_train_n.shape[1:])

unet.summary()

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=20,

        restore_best_weights=True

    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=6,

        min_lr=1e-6,

        verbose=1

    ),

    tf.keras.callbacks.ModelCheckpoint(

        "best_unet.keras",

        monitor="val_loss",

        save_best_only=True

    )

]

In [ ]:
history = unet.fit(

    X_train_n,

    Y_train_n,

    validation_data=(X_val_n, Y_val_n),

    epochs=200,

    batch_size=4, 

    callbacks=callbacks,

    shuffle=True

)

## unet plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hist = history.history
epochs = np.arange(1, len(hist["loss"]) + 1)


# =========================
# Huber loss
# =========================
best_loss_epoch = np.argmin(hist["val_loss"]) + 1
best_val_loss = np.min(hist["val_loss"])

plt.figure(figsize=(8, 5))


plt.plot(
    epochs,
    hist["loss"],
    label="Training Huber loss"
)

plt.plot(
    epochs,
    hist["val_loss"],
    label="Validation Huber loss"
)

plt.axvline(
    best_loss_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best epoch = {best_loss_epoch}"
)

plt.scatter(
    best_loss_epoch,
    best_val_loss,
    s=60,
    zorder=5
)

plt.xlabel("Epoch")
plt.ylabel("Huber loss")
plt.title("Training and validation Huber loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# =========================
# Mean absolute error
# =========================
best_mae_epoch = (
    np.argmin(hist["val_mean_absolute_error"]) + 1
)

best_val_mae = np.min(
    hist["val_mean_absolute_error"]
)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    hist["mean_absolute_error"],
    label="Training MAE"
)

plt.plot(
    epochs,
    hist["val_mean_absolute_error"],
    label="Validation MAE"
)

plt.axvline(
    best_mae_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best epoch = {best_mae_epoch}"
)

plt.scatter(
    best_mae_epoch,
    best_val_mae,
    s=60,
    zorder=5
)

plt.xlabel("Epoch")
plt.ylabel("Mean absolute error")
plt.title("Training and validation MAE")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# =========================
# Root mean squared error
# =========================
best_rmse_epoch = (
    np.argmin(hist["val_root_mean_squared_error"]) + 1
)

best_val_rmse = np.min(
    hist["val_root_mean_squared_error"]
)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    hist["root_mean_squared_error"],
    label="Training RMSE"
)

plt.plot(
    epochs,
    hist["val_root_mean_squared_error"],
    label="Validation RMSE"
)

plt.axvline(
    best_rmse_epoch,
    linestyle="--",
    alpha=0.7,
    label=f"Best epoch = {best_rmse_epoch}"
)

plt.scatter(
    best_rmse_epoch,
    best_val_rmse,
    s=60,
    zorder=5
)

plt.xlabel("Epoch")
plt.ylabel("Root mean squared error")
plt.title("Training and validation RMSE")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


print("Best validation results")
print("-----------------------")
print(
    f"Huber loss: {best_val_loss:.4f} "
    f"at epoch {best_loss_epoch}"
)
print(
    f"MAE:        {best_val_mae:.4f} "
    f"at epoch {best_mae_epoch}"
)
print(
    f"RMSE:       {best_val_rmse:.4f} "
    f"at epoch {best_rmse_epoch}"
)

In [ ]:
print(len(Y_true_maps))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Predict validation set
Y_pred_n = unet.predict(
    X_val_n,
    batch_size=4,
    verbose=1
)


# Remove final channel dimension if present
if Y_val_n.ndim == 4 and Y_val_n.shape[-1] == 1:
    Y_true_maps = Y_val_n[..., 0]
else:
    Y_true_maps = np.asarray(Y_val_n)

if Y_pred_n.ndim == 4 and Y_pred_n.shape[-1] == 1:
    Y_pred_maps = Y_pred_n[..., 0]
else:
    Y_pred_maps = np.asarray(Y_pred_n)


# Show three representative samples
#sample_indices = [
 #   0,
 #   len(Y_true_maps) // 2,
  #  len(Y_true_maps) - 1
#]
sample_indices = [0,2,4,6,7,8,10, 12,14]


for sample_index in sample_indices:

    true_map = Y_true_maps[sample_index]
    predicted_map = Y_pred_maps[sample_index]
    error_map = predicted_map - true_map

    sample_mae = np.mean(
        np.abs(error_map)
    )

    sample_rmse = np.sqrt(
        np.mean(error_map**2)
    )

    # Same scale for truth and prediction
    value_limit = np.max(
        np.abs(
            np.concatenate([
                true_map.ravel(),
                predicted_map.ravel()
            ])
        )
    )

    # Robust scale for the error map
    error_limit = np.percentile(
        np.abs(error_map),
        99
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 4.5)
    )

    im0 = axes[0].imshow(
        true_map,
        origin="lower",
        cmap="RdBu_r",
        vmin=-value_limit,
        vmax=value_limit
    )

    axes[0].set_title("True correction")
    axes[0].set_xlabel("Longitude pixel")
    axes[0].set_ylabel("Latitude pixel")

    plt.colorbar(
        im0,
        ax=axes[0],
        fraction=0.046,
        pad=0.04
    )

    im1 = axes[1].imshow(
        predicted_map,
        origin="lower",
        cmap="RdBu_r",
        vmin=-value_limit,
        vmax=value_limit
    )

    axes[1].set_title("Predicted correction")
    axes[1].set_xlabel("Longitude pixel")
    axes[1].set_ylabel("Latitude pixel")

    plt.colorbar(
        im1,
        ax=axes[1],
        fraction=0.046,
        pad=0.04
    )

    im2 = axes[2].imshow(
        error_map,
        origin="lower",
        cmap="RdBu_r",
        vmin=-error_limit,
        vmax=error_limit
    )

    axes[2].set_title(
        "Prediction − truth\n"
        f"MAE = {sample_mae:.3f}, "
        f"RMSE = {sample_rmse:.3f}"
    )

    axes[2].set_xlabel("Longitude pixel")
    axes[2].set_ylabel("Latitude pixel")

    plt.colorbar(
        im2,
        ax=axes[2],
        fraction=0.046,
        pad=0.04
    )

    fig.suptitle(
        f"Validation sample {sample_index}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Change this if your LWC is in another channel
lwc_channel = 4

for sample_index in sample_indices:

    lwc = X_val_n[sample_index, :, :, lwc_channel]

    plt.figure(figsize=(6,5))

    im = plt.imshow(
        lwc,
        origin="lower",
        cmap="viridis"     # same colour map as before
    )

    plt.title(f"LWC\nValidation sample {sample_index}")
    plt.xlabel("Longitude pixel")
    plt.ylabel("Latitude pixel")

    plt.colorbar(
        im,
        label="Normalized LWC"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


Y_true_flat = Y_true_maps.reshape(-1)
Y_pred_flat = Y_pred_maps.reshape(-1)

residuals = Y_pred_flat - Y_true_flat


# Remove invalid values
valid = (
    np.isfinite(Y_true_flat)
    & np.isfinite(Y_pred_flat)
)

Y_true_flat = Y_true_flat[valid]
Y_pred_flat = Y_pred_flat[valid]
residuals = residuals[valid]


# Overall validation metrics
overall_mae = np.mean(
    np.abs(residuals)
)

overall_rmse = np.sqrt(
    np.mean(residuals**2)
)

mean_bias = np.mean(
    residuals
)

correlation = np.corrcoef(
    Y_true_flat,
    Y_pred_flat
)[0, 1]

ss_res = np.sum(
    (Y_true_flat - Y_pred_flat)**2
)

ss_tot = np.sum(
    (Y_true_flat - np.mean(Y_true_flat))**2
)

r_squared = 1 - ss_res / ss_tot


print("Overall validation performance")
print("------------------------------")
print(f"MAE:         {overall_mae:.4f}")
print(f"RMSE:        {overall_rmse:.4f}")
print(f"Mean bias:   {mean_bias:.4f}")
print(f"Correlation: {correlation:.4f}")
print(f"R²:          {r_squared:.4f}")


# Random subset for plotting
max_points = 50000

if len(Y_true_flat) > max_points:

    rng = np.random.default_rng(42)

    selected = rng.choice(
        len(Y_true_flat),
        size=max_points,
        replace=False
    )

    true_plot = Y_true_flat[selected]
    pred_plot = Y_pred_flat[selected]

else:

    true_plot = Y_true_flat
    pred_plot = Y_pred_flat


plot_min = min(
    true_plot.min(),
    pred_plot.min()
)

plot_max = max(
    true_plot.max(),
    pred_plot.max()
)


plt.figure(figsize=(7, 7))

plt.hexbin(
    true_plot,
    pred_plot,
    gridsize=80,
    mincnt=1,
    bins="log",
    cmap="viridis"
)

plt.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("True correction")
plt.ylabel("Predicted correction")

plt.title(
    "True versus predicted correction\n"
    f"R² = {r_squared:.3f}, "
    f"correlation = {correlation:.3f}"
)

plt.colorbar(
    label="Logarithmic pixel density"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.hist(
    residuals,
    bins=100,
    density=True,
    alpha=0.8
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Zero error"
)

plt.axvline(
    mean_bias,
    linestyle=":",
    linewidth=2,
    label=f"Mean bias = {mean_bias:.3f}"
)

plt.xlabel("Prediction error: prediction − truth")
plt.ylabel("Probability density")
plt.title("Validation residual distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Saving model 